# 06 - Structured Streaming restart and event-time reasoning

Prerequisites / execution context: Run 00. Uses a dedicated append-only Delta source and AvailableNow. No permanently running query.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R29: Structured Streaming concepts](https://docs.databricks.com/aws/en/structured-streaming/concepts); [R30: Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints); [R31: Watermarks](https://docs.databricks.com/aws/en/structured-streaming/watermarks); [R04: Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Create a source, sink, and checkpoint
This demonstrates incremental append processing. Query history belongs to the checkpoint, not to the notebook Python process.


In [ ]:
import uuid
rid=uuid.uuid4().hex[:10]
SRC=f"{BRONZE}.stream_src_{rid}"; DST=f"{SILVER}.stream_dst_{rid}"
CP=f"{VOLUME}/streams/{rid}/checkpoint"
spark.sql(f"CREATE TABLE {SRC} (id INT,event_ts TIMESTAMP) USING DELTA")
spark.sql(f"INSERT INTO {SRC} VALUES (1,TIMESTAMP '2026-11-01 10:00:00')")
def drain():
    q=(spark.readStream.table(SRC).writeStream.option("checkpointLocation",CP)
       .trigger(availableNow=True).toTable(DST))
    q.awaitTermination()
    return q
q=drain()
assert spark.table(DST).count()==1
drain()
assert spark.table(DST).count()==1
spark.sql(f"INSERT INTO {SRC} VALUES (2,TIMESTAMP '2026-11-01 10:01:00')")
drain()
assert spark.table(DST).count()==2


## 2. Build a watermark query plan without running a second stream
A 10-minute watermark is relative to observed event time, not wall-clock time. Events within the supported lateness threshold are protected; older ones are not guaranteed to be processed. Official Databricks documentation: [R31: Watermarks](https://docs.databricks.com/aws/en/structured-streaming/watermarks).


In [ ]:
stateful=(spark.readStream.table(SRC).withWatermark("event_ts","10 minutes")
          .groupBy(F.window("event_ts","5 minutes")).count())
stateful.explain()
# A real writer for this different query must use a DIFFERENT checkpoint.


## Challenges
1. Source max event time is 12:20 and lateness is 10 minutes: reason about 12:15 and 12:01 events. The first is within the threshold; the second is outside the guaranteed range. Actual watermark advancement occurs across batches.
2. Why not use this append stream unchanged after UPDATE on the source? Append-only processing is not a general CDC solution. Study CDF/AUTO CDC in lab 12.
3. Why not switch this checkpoint to a different aggregation? Stateful query changes have restart compatibility restrictions. Official Databricks documentation: [R30: Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints); [R68: Change data feed](https://docs.databricks.com/aws/en/tables/features/change-data-feed).
